# Quick Model Test - Trained Deepfake Detectors

Fast validation notebook converted from test script.

**What this does:**
- ✓ Loads PyTorch 2.0.0+cpu
- ✓ Loads SegFormer forensic model
- ✓ Loads trained Temporal LSTM
- ✓ Tests inference with dummy data

Run all cells sequentially to verify models work.

In [ ]:
# Cell 1: Core imports

import os
import sys
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 60)
print("Testing Trained Deepfake Detection Models")
print("=" * 60)

print("\n[1/6] Loading basic libraries...")
print(f"✓ Python {sys.version.split()[0]}")
print(f"✓ NumPy {np.__version__}")
print(f"✓ OpenCV {cv2.__version__}")
print(f"✓ Pandas {pd.__version__}")

In [ ]:
# Cell 2: PyTorch check

print("\n[2/6] Loading PyTorch...")
try:
    import torch
    import torch.nn as nn
    import torchvision.transforms as T
    print(f"✓ PyTorch {torch.__version__}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✓ Device: {device}")
    torch_ok = True
except Exception as e:
    print(f"✗ PyTorch import failed: {type(e).__name__}")
    print(f"  {str(e)[:200]}")
    torch_ok = False
    raise

print("PyTorch loaded successfully!")

In [ ]:
# Cell 3: Model paths

print("\n[3/6] Checking model files...")
BASE_DIR = Path.cwd()
MODELS_DIR = Path(r'c:\Users\hp\Downloads')

TEMPORAL_MODEL_PATH = MODELS_DIR / 'deepfake_temporal_medium_lstm.pth'
BASELINE_MODEL_PATH = MODELS_DIR / 'deepfake_baseline_medium.pkl'
SEGFORMER_CODE_DIR = BASE_DIR
SEGFORMER_WEIGHTS = BASE_DIR / 'models' / 'face_segmentation_kaggle_model.pth'

print(f"  Temporal (.pth): {'✓' if TEMPORAL_MODEL_PATH.exists() else '✗'} {TEMPORAL_MODEL_PATH}")
print(f"  Baseline (.pkl): {'✓' if BASELINE_MODEL_PATH.exists() else '✗'} {BASELINE_MODEL_PATH}")
print(f"  SegFormer weights: {'✓' if SEGFORMER_WEIGHTS.exists() else '✗'} {SEGFORMER_WEIGHTS}")

In [ ]:
# Cell 4: Load SegFormer

print("\n[4/6] Loading SegFormer model...")
if str(SEGFORMER_CODE_DIR) not in sys.path:
    sys.path.append(str(SEGFORMER_CODE_DIR))

from segformer_model import SegformerEdgeAware

segformer = SegformerEdgeAware(num_classes=11, pretrained=True).to(device)

if SEGFORMER_WEIGHTS.exists():
    state = torch.load(SEGFORMER_WEIGHTS, map_location=device)
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    segformer.load_state_dict(state, strict=False)
    print("✓ SegFormer loaded with trained weights")
else:
    print("⚠ Using base pretrained SegFormer (no fine-tuned weights)")

segformer.eval()
print("SegFormer ready!")

In [ ]:
# Cell 5: Temporal model architecture

class TemporalDeepfakeDetector(nn.Module):
    """LSTM-based binary detector over ordered frame features."""
    def __init__(self, num_features=13, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        bsz, steps, feat = x.shape
        encoded = self.encoder(x.reshape(bsz * steps, feat))
        encoded = encoded.reshape(bsz, steps, -1)
        _, (h_n, _) = self.lstm(encoded)
        last_hidden = h_n[-1]
        logits = self.head(last_hidden)
        return logits.squeeze(-1)

print("✓ Temporal model architecture defined")

In [ ]:
# Cell 6: Load trained models

print("\n[5/6] Loading trained models...")

# Temporal model
temporal_model = None
if TEMPORAL_MODEL_PATH.exists():
    try:
        state = torch.load(TEMPORAL_MODEL_PATH, map_location=device)
        temporal_model = TemporalDeepfakeDetector(num_features=13, hidden_dim=64).to(device)
        temporal_model.load_state_dict(state)
        temporal_model.eval()
        print("✓ Temporal LSTM model loaded")
    except Exception as e:
        print(f"✗ Temporal model load failed: {e}")

# Baseline model
baseline_model = None
if BASELINE_MODEL_PATH.exists():
    try:
        import joblib
        baseline_model = joblib.load(BASELINE_MODEL_PATH)
        print("✓ Baseline model loaded")
    except Exception as e:
        print(f"⚠ Baseline model load failed: {type(e).__name__}")
        print(f"  (Expected due to version mismatch)")

print(f"\nModels ready:")
print(f"  Temporal: {'✓ YES' if temporal_model is not None else '✗ NO'}")
print(f"  Baseline: {'✓ YES' if baseline_model is not None else '✗ NO'}")

In [ ]:
# Cell 7: Test inference

print("\n[6/6] Testing inference with dummy data...")

if temporal_model is not None:
    try:
        dummy_sequence = torch.randn(1, 24, 13).to(device)
        with torch.no_grad():
            logits = temporal_model(dummy_sequence)
            prob = torch.sigmoid(logits).item()
        print(f"✓ Temporal model inference: prob={prob:.4f}")
        print(f"  (Random input, so prediction is meaningless - just testing forward pass)")
    except Exception as e:
        print(f"✗ Temporal inference failed: {e}")
else:
    print("⚠ Temporal model not available for testing")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"PyTorch: ✓ Working")
print(f"SegFormer: ✓ Working")
print(f"Temporal Model: {'✓ Loaded & Working' if temporal_model is not None else '✗ Not Available'}")
print(f"Baseline Model: {'✓ Loaded' if baseline_model is not None else '⚠ Version Mismatch'}")

if temporal_model is not None:
    print("\n✅ MODEL VALIDATION SUCCESSFUL!")
    print("\nYou can now use the temporal model for predictions.")
    print("Next: Run the full test_trained_models.ipynb to test on real videos.")
else:
    print("\n⚠ Model validation incomplete - check errors above.")

## ✅ Validation Complete

If all cells ran successfully:
- ✓ PyTorch 2.0.0+cpu is working
- ✓ SegFormer model loaded
- ✓ Temporal LSTM loaded from trained weights
- ✓ Forward pass works (inference ready)

**Next steps:**
1. Use the full [test_trained_models.ipynb](test_trained_models.ipynb) to test on real videos
2. Test videos are available in: `test_videos/`
3. Or record your own with webcam/phone and test!

**Model performance:**
- Trained on Kaggle with ~160 videos per class
- Best validation AUC: ~0.63
- Early stopping at epoch 19/22